# Heart, kidney and patient outcome-model experiments

The external classification notebook is a layout reference, not instructions. This notebook follows its preprocessing → pipeline → GridSearchCV → best_params_ → classification report/confusion matrix workflow.

**Meaning of outputs:** `heart_risk` and `kidney_risk` predict recorded death from organ-related feature subsets; the dataset has no organ-specific disease/outcome targets. `patient_risk` uses all approved baseline features, avoiding an arbitrary average of correlated scores. All three scores are uncalibrated; lower/middle/higher bands are relative development-score thirds, not clinical organ severity. Existing EF/creatinine indicators stay separate. Source: [UCI](https://archive.ics.uci.edu/dataset/519/heart+failure+clinical+records). Run setup first; Databricks requires the full repository.

Feature groups are now literature-informed (`organ-feature-groups-v2`): heart = EF, age, hypertension, diabetes, smoking, anaemia; kidney = creatinine, age, sex, diabetes, hypertension, smoking. See [medical rationale and sources](../../docs/feature-selection.md). These are candidate assignments, not validated organ-specific predictors. Rerun to replace old feature-group artifacts. Previous test-set runs mean subsequent comparisons are exploratory rather than fresh independent validation.

In [ ]:
# Locate the full repository before importing its src-layout Python package.
# Local VS Code/Jupyter: leave this empty when running inside the repository.
# Databricks/serverless or another working directory: set the actual absolute
# repository folder containing pyproject.toml, src/, and data/ (not this notebook).
from pathlib import Path
import os
import sys

REPO_ROOT_OVERRIDE = ""  # Example: "/Workspace/Users/<you>/H.A.K-Industry-Hackathon"
configured_root = REPO_ROOT_OVERRIDE or os.environ.get("HF_REPO_ROOT", "")

if configured_root:
    REPO_ROOT = Path(configured_root).expanduser().resolve()
else:
    working_dir = Path.cwd().resolve()
    REPO_ROOT = next(
        (candidate for candidate in (working_dir, *working_dir.parents)
         if (candidate / "pyproject.toml").is_file()
         and (candidate / "src/hf_followup/__init__.py").is_file()),
        None,
    )

if REPO_ROOT is None or not (REPO_ROOT / "src/hf_followup/__init__.py").is_file():
    raise FileNotFoundError(
        "The full repository was not found. Set REPO_ROOT_OVERRIDE to the folder "
        "containing src/hf_followup and data. Importing only this .ipynb into "
        "Databricks does not include the shared Python source; clone/upload the full repo."
    )

SOURCE_DIR = REPO_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

import hf_followup
SOURCE_CSV = REPO_ROOT / "data/heart_failure_clinical_records.csv"
if not SOURCE_CSV.is_file():
    raise FileNotFoundError(f"Bundled dataset missing: {SOURCE_CSV}. Include the repo's data folder.")
print(f"Package imported from: {hf_followup.__file__}")
print(f"Dataset: {SOURCE_CSV}")
print(f"Notebook Python: {sys.executable}")


In [ ]:
# Install missing packages in this notebook environment if necessary:
# %pip install -e "{REPO_ROOT}[ml]"
# %pip install matplotlib==3.10.7
from hf_followup.data.ingest import ingest_csv
from hf_followup.ml.experiments import run_experiments
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ingestion = ingest_csv(SOURCE_CSV)
raw = pd.read_csv(SOURCE_CSV)
# Ingestion validates numeric/binary/range constraints, separates outcomes and preserves IDs.
# Drop incomplete/quarantine malformed records; retain plausible extremes and feature duplicates.
quality = {"source_rows": len(raw), "accepted_rows": ingestion.cohort.manifest["accepted_count"],
           "missing_rows": ingestion.cohort.manifest["missing_rows"], "missing_cells": ingestion.cohort.manifest["missing_cells"],
           "excluded_rows": ingestion.cohort.manifest["excluded_count"], "exact_source_duplicates": int(raw.duplicated().sum()),
           "duplicate_feature_rows": int(raw.drop(columns=["time", "DEATH_EVENT"]).duplicated(keep=False).sum())}
display(pd.Series(quality, name="Cleaning audit"))
display(raw.describe().T)
display(pd.DataFrame(ingestion.cohort.manifest["warnings"]))
# Bundled data require no row deletion or imputation. Missing predictors in later uploads
# need a separately versioned fold-fitted imputer; do not silently change this population.

## Feature selection — edit the lists below

These explicit lists are passed into the model-fitting function. Medical rationale and source links: [feature selection notes](../../docs/feature-selection.md).

In [ ]:
# FEATURE SELECTION: these lists directly control the columns fitted for each model.
# Edit here for experiments. run_experiments(..., feature_groups=GROUPS) uses these lists.
# Medically informed candidates; target remains DEATH_EVENT for all three outputs.

# Heart: pumping measurement + cardiovascular risk/anaemia context (AHA/NHLBI).
heart_features = [
    "ejection_fraction", "age", "high_blood_pressure", "diabetes", "smoking", "anaemia",
]

# Kidney: filtration-related marker + age/sex interpretation + renal risk context (NKF/NIDDK).
# Sodium and anaemia are excluded from this focused subset; both remain in patient_features.
kidney_features = [
    "serum_creatinine", "age", "sex", "diabetes", "high_blood_pressure", "smoking",
]

# Patient: all 11 approved baseline measurements; no time or death label in predictors.
patient_features = [
    "age", "anaemia", "creatinine_phosphokinase", "diabetes", "ejection_fraction",
    "high_blood_pressure", "platelets", "serum_creatinine", "serum_sodium", "sex", "smoking",
]

GROUPS = {"heart_risk": heart_features, "kidney_risk": kidney_features, "patient_risk": patient_features}
display(pd.DataFrame({name: pd.Series(fields) for name, fields in GROUPS.items()}))


## Three tasks, three model families

Logistic regression uses scaled numeric fields; random forest and **gradient boosting** use the same explicit column selection without scaling. Every model is a preprocessor+model Pipeline. `GridSearchCV.fit` fits candidates inside five stratified development folds and refits `best_estimator_` on the 239 development records. The 60 test records never select parameters, family, bands or thresholds. Previous inspection of this test set makes comparisons exploratory.

The earlier four-candidate grids were too narrow for broad tuning. The editable grids below now cover 10 logistic, 24 forest and 16 boosting configurations per output (750 CV fits total, plus refits/OOF fits). Shallow boosting trees and smaller forest leaves test useful capacity/regularization tradeoffs on this small dataset. Very large grids can overfit cross-validation as well as take longer.

**Current selection rule:** grid parameters maximize development positive-class F1. Each tuned family receives a development-OOF-F1 threshold, then families are compared on the same 239 development patients: **fewest false negatives first, most true negatives second, most true positives third**. At fixed positive counts, fewer FN already means more TP. Remaining ties use CV AUC then simpler family. The selection table below shows these actual development confusion counts.

This gives FN priority; it is not a guarantee that FN and TN can both improve. Thresholds retain F1 tuning to balance precision/recall rather than lowering them until everyone is positive. `MODEL_SELECTION="cv_metric"` and `REFIT_METRIC="capture_at_k"` restore the original PRD queue objective. Test matrices never choose the winner.

Classification is separate from ranking: reports show default 0.5 and a threshold maximizing positive-class F1 on development out-of-fold predictions. These predictions are conditional on hyperparameter selection, not nested independent validation. Lowering a threshold may recover positives while adding false positives; it does not improve ranking or calibrate probabilities. See [scikit-learn threshold guidance](https://scikit-learn.org/stable/modules/classification_threshold.html).

Band cutoffs remain development OOF-score tertiles. Test K=ceil((25/299)*60)=6. Relative bands are an experimental contract distinct from PRD queue priority labels; organ colors remain separate.


In [ ]:
# EDITABLE SEARCH CONFIGURATION: these exact grids are passed to GridSearchCV.
# Restart the kernel and Run All after changing the shared helper to reload its code.
REFIT_METRIC = "f1_positive"  # Parameter tuning: balance death-class precision and recall.
MODEL_SELECTION = "min_false_negatives"  # Family selection: fewest OOF FN, then most TN/TP.
# To restore queue-based selection: REFIT_METRIC="capture_at_k", MODEL_SELECTION="cv_metric".
PARAM_GRIDS = {
    "logistic_regression": {
        "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
        "model__class_weight": [None, "balanced"],
    },
    "random_forest": {
        # n_estimators=150 fixed: tune capacity/feature sampling and imbalance first.
        "model__max_depth": [4, 8, None],
        "model__min_samples_leaf": [1, 5],
        "model__max_features": ["sqrt", 1.0],
        "model__class_weight": [None, "balanced"],
    },
    "gradient_boosting": {
        # Shallow sequential trees; subsample=0.8 fixed. No extra dependency needed.
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [1, 2],
        "model__min_samples_leaf": [3, 8],
    },
}
from sklearn.model_selection import ParameterGrid
print("Refit objective:", REFIT_METRIC)
print("Candidates per output:", {family: len(ParameterGrid(grid)) for family, grid in PARAM_GRIDS.items()})


In [ ]:
# Edit the parameter grids in the preceding cell, then rerun this cell and those below.
# Choose using development confusion counts; test reports cannot change the winner.
# Refresh the shared helper even when this kernel imported its older version earlier.
# Re-running "from ... import run_experiments" alone does not reload cached module code.
import importlib
import inspect
import hf_followup.ml.experiments as experiment_module

expected_helper = (SOURCE_DIR / "hf_followup/ml/experiments.py").resolve()
if Path(experiment_module.__file__).resolve() != expected_helper:
    raise RuntimeError(
        f"The kernel loaded {experiment_module.__file__}, but this notebook uses {expected_helper}. "
        "Restart the kernel and run the repository setup cell first."
    )
importlib.invalidate_caches()
experiment_module = importlib.reload(experiment_module)
run_experiments = experiment_module.run_experiments  # Replace the stale function reference.
if not {"param_grids", "refit_metric", "selection_policy"}.issubset(inspect.signature(run_experiments).parameters):
    raise RuntimeError(
        f"The helper at {experiment_module.__file__} is an older version. "
        "Update the full repository (including src/hf_followup/ml/experiments.py), "
        "then rerun this cell; updating only the notebook is insufficient."
    )
print(f"Reloaded experiment helper: {experiment_module.__file__}")

results = run_experiments(
    ingestion, REPO_ROOT / "runtime/ml-experiments",
    feature_groups=GROUPS, param_grids=PARAM_GRIDS, refit_metric=REFIT_METRIC,
    selection_policy=MODEL_SELECTION,
)
for task, searches in results["searches"].items():
    print(f"\n{task}: selected {results['report']['tasks'][task]['selected_family']} using {MODEL_SELECTION}")
    for family, grid_search in searches.items():
        print(family, "best_params_:", grid_search.best_params_, "best_score_:", grid_search.best_score_)
        table = pd.DataFrame(results["report"]["tasks"][task]["models"][family]["cv_results"])
        display(table.sort_values(f"mean_test_{REFIT_METRIC}", ascending=False))

# Show the development counts that actually chose each winner (not the test matrix).
selection_table = []
for task, task_report in results["report"]["tasks"].items():
    for family, model_report in task_report["models"].items():
        selection_table.append({"task": task, "family": family,
            "selected": family == task_report["selected_family"],
            "threshold": model_report["development_threshold"],
            **model_report["development_oof"]["counts"]})
display(pd.DataFrame(selection_table))
# The selected classifier includes its threshold; .predict() alone still uses 0.5.
best_heart_model = results["selected_models"]["heart_risk"]
best_kidney_model = results["selected_models"]["kidney_risk"]
best_patient_model = results["selected_models"]["patient_risk"]
best_thresholds = results["selected_thresholds"]
# For new validated baseline rows: predictions = (model.predict_proba(rows[features])[:, 1] >= threshold).


In [ ]:
# Compare all three families at default and development-tuned classification thresholds.
# Rows = actual [no death, death]; columns = predicted [no death, death].
# Lower-left = missed recorded deaths; upper-right = false positive alerts.
comparison = []
for task, task_report in results["report"]["tasks"].items():
    fig, axes = plt.subplots(2, len(task_report["models"]), figsize=(18, 9), squeeze=False)
    for col, (family, model_report) in enumerate(task_report["models"].items()):
        print(f"\n{task} / {family} — exploratory held-out test")
        display(pd.Series(model_report["test"], name="Ranking and discrimination"))
        for row, (label, threshold, metrics) in enumerate([
            ("default", 0.5, model_report),
            ("development-tuned", model_report["development_threshold"], model_report["thresholded"]),
        ]):
            print(f"{label} threshold = {threshold:.3f}")
            display(pd.DataFrame(metrics["classification_report"]).T)
            ConfusionMatrixDisplay(np.asarray(metrics["confusion_matrix"]), display_labels=["No death", "Death"]).plot(ax=axes[row, col], cmap="Blues", colorbar=False)
            axes[row, col].set_title(f"{family} / {label} ({threshold:.3f})")
            positive = metrics["classification_report"]["recorded_death"]
            comparison.append({"task": task, "family": family, "threshold_policy": label,
                               "threshold": threshold, "precision": positive["precision"],
                               "recall": positive["recall"], "f1": positive["f1-score"],
                               "test_auc": model_report["test"]["roc_auc"],
                               "cv_selection_score": model_report["best_cv_score"]})
    fig.suptitle(task)
    plt.tight_layout()
    plt.show()
display(pd.DataFrame(comparison))  # Diagnostic only; never choose a winner from test F1.


In [ ]:
# Shared program handoff: three scores, relative bands, provenance and measurement indicators.
display(results["predictions"].head(15))
for task, report in results["report"]["tasks"].items():
    print(task, report["band_cutoffs"], report["calibration_status"])
assert "DEATH_EVENT" not in results["predictions"].columns
assert "time" not in results["predictions"].columns
print("Exports: runtime/ml-experiments/risk_outputs.csv, experiment_report.json, split_manifest.json and three fitted pipelines.")

## Program integration and remaining cloud work

The selected pipelines now have a standalone app handoff. Run `make publish-models`
in the training environment to freeze this notebook's results into a versioned JSON
cache. Alternatively, `make train` refits `configs/ml_selection.json` without another
search or family change. See [ML integration](../../docs/ml-integration.md).

`RiskPredictor` in `src/hf_followup/ml/inference.py` scores new baseline records;
`scripts/predict.py` supplies batch JSON inference. The API loads cached predictions
at startup and exposes all three scores/bands/thresholds to frontend and 3D interfaces.
It never fits or deserializes model files during interactions.

MLflow logging/signatures, verified workspace Delta publication, SQL transport and
clinical validation remain owner work. Notebook 03 supplies gated publication entry
points; do not claim a live cloud integration until it is verified. Organ-feature
outcome proxies remain distinct from organ-specific disease labels and measurements.
